# 03 · Few-shot 예시와 편향

Few-shot 프롬프트는 답을 내기 전에 **예시 몇 개**를 함께 보여주는 방식이다. 이 노트북에서는 예시가 언제 **도움**이 되고 언제 **편향이나 과적합**을 부르는지 직접 실행해 확인한다.
모든 실험은 `research_utils`의 헬퍼 함수(`ask`, `ask_json`, `compare`, `keyword_hits`)만 사용한다.

이 노트북이 검증하는 팁:

- **Tip 8 · JSON 예시의 값 편향**: 값까지 채워 넣은 예시를 주면, 실제 입력에 그 정보가 없어 빈칸이어야 할 때도 모델이 예시의 값을 그대로 가져온다.
- **Tip 9 · 멀티샷 포맷**: 예시를 `문제→정답`으로 주는 것보다 `<scratchpad>사고</scratchpad><answer>정답</answer>` 구조로 주면, 모델도 새 문제에서 먼저 사고하고 답을 내는 흐름을 따라온다.
- **Tip 23 · 예시 과적합 판별**: 예시에 심어 둔 독특한 단어가 전혀 무관한 답변에까지 반복되면, 모델이 예시의 내용에 과적합됐다는 신호다.
- **Tip 26 · 구형 모델 내구성 테스트**: 값싼 구형 모델(gpt-4.1-nano)에서도 프롬프트가 의도대로 동작하면, 그 프롬프트는 특정 모델의 성능에 기대지 않는 튼튼한 구조라는 뜻이다.

> gpt-5-nano 실측 제약: `temperature`가 1로 고정돼 있어(창의성은 프롬프트 텍스트로만 조절), `max_tokens`를 지원하지 않아 `max_completion_tokens`를 쓰고, `reasoning_effort`와 `verbosity`를 지원한다. 추론(reasoning) 모델이라 `max_completion_tokens`를 너무 작게 주면 본문이 빈 문자열로 나올 수 있어 넉넉히 준다.

> 셀은 **실행되지 않은 상태**로 저장돼 있다. 위에서부터 순서대로 직접 Run 하면 된다.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## Tip 8 · JSON 예시에 값을 채우면 생기는 편향

**요지.** 정보 추출용 JSON 예시에 **값까지 다 채워** 주면, 모델은 그 값을 참고할 정답처럼 받아들인다. 그래서 실제 입력에는 그 정보가 **없어서 빈칸(null)이어야 하는 경우에도** 예시에 있던 값을 그대로 가져온다.

**무엇을 어떻게 검증하나.** 아래 입력에는 `order_id`와 `refund_amount` 정보가 **전혀 없다**. 이 상태에서 예시만 다르게 준다.
- **A) 값이 채워진 예시**: 예시에 `order_id="A7788"`, `refund_amount=50` 같은 실제 값이 들어 있다.
- **B) 빈 스키마 + 미러링 금지 문구**: 값은 모두 `null`로 비워 구조만 보여주고, "예시는 구조를 보여줄 뿐이니 값을 따라 쓰지 말라"는 문구를 영어로 함께 넣는다. (미러링 = 예시 값을 그대로 복사하는 것)

`ask_json`으로 두 경우를 비교해, **A가 예시 값(A7788/50)을 가져오는지**, **B는 없는 필드를 `null`로 두는지** 관찰한다.

In [ ]:
# --- Tip 8 실험: 값이 없어야 할 필드를 예시에서 베끼는가? ---
schema_keys = ["customer_name", "order_id", "issue_type", "refund_amount"]

# 정답이 없는 입력: 이름/이슈만 있고 주문번호·환불금액은 언급이 전혀 없음
target_input = "Hi, this is Minjun. The app keeps crashing when I open the settings page. Please fix it."

# A) 값이 꽉 찬 예시 (답안지) — 예시 값을 베낄 위험
prompt_A = f"""Extract fields into a JSON object with keys {schema_keys}.

Example:
Input: "Hello, I am Sarah, order #A7788. The package arrived broken, I want $50 back."
Output: {{"customer_name": "Sarah", "order_id": "A7788", "issue_type": "damaged_package", "refund_amount": 50}}

Now extract from this input:
Input: "{target_input}"
Output:"""

# B) 빈 스키마(키만) + 미러링 금지 경고
prompt_B = f"""Extract fields into a JSON object with keys {schema_keys}.

The example below shows STRUCTURE ONLY. Do NOT copy or mirror any example values.
If a field is not present in the input, set it to null.

Schema example (structure only, values are placeholders):
{{"customer_name": null, "order_id": null, "issue_type": null, "refund_amount": null}}

Now extract from this input:
Input: "{target_input}"
Output:"""

a = ask_json(prompt_A)
b = ask_json(prompt_B)
compare("A) 값이 꽉 찬 예시 (베낄 위험)", str(a), "B) 빈 스키마 + 미러링 금지 경고", str(b))

print("\n[관찰 포인트]")
print("- A의 order_id/refund_amount 가 예시값 'A7788'/50 을 베꼈다면 → 답안지 편향 발생")
print("- B의 order_id/refund_amount 가 None/null 이면 → 미러링 금지 경고가 작동")

## Tip 9 · 멀티샷 포맷 — 사고 과정을 예시에 담기

**요지.** 예시를 `문제→정답` 형태로만 주면, 모델은 사고 과정 없이 답만 바로 내놓는 흐름을 따라 한다. 반대로 예시를 `<scratchpad>사고 과정</scratchpad><answer>정답</answer>` 구조로 주면, 모델도 새 문제에서 **먼저 사고하고(scratchpad) 답을 내는(answer)** 순서를 따라온다.

**무엇을 어떻게 검증하나.** 같은 산수 문제를 두고 예시 형식만 다르게 준다.
- **A) `문제→정답` 예시**: 사고 과정 없이 답만 보여준다.
- **B) `scratchpad/answer` 예시**: 풀이 과정과 답을 나눈 구조로 보여준다.

그런 다음 정답이 분명한 새 문제를 주고, 두 출력에서 **풀이 과정이 드러나는지**와 **정답을 맞히는지**를 비교한다. (모델의 자체 추론 능력이 아니라 예시 형식의 효과만 보려고 `reasoning_effort="minimal"`로 낮춰 둔다.)

In [ ]:
# --- Tip 9 실험: 예시 포맷이 새 문제의 추론 흐름을 바꾸는가? ---
new_problem = "A shop sells pens at 3 for $2. Ravi buys 12 pens and pays with a $10 bill. How much change does he get?"

# A) 문제 → 정답 (사고과정 없이 답만)
prompt_A = f"""Solve the problem.

Q: A train travels 60 km in 1.5 hours. What is its average speed?
A: 40 km/h

Q: A box has 24 apples. You remove one third. How many remain?
A: 16

Q: {new_problem}
A:"""

# B) scratchpad(사고) → answer 구조를 예시에 이식
prompt_B = f"""Solve the problem. Follow the exact format of the examples.

Q: A train travels 60 km in 1.5 hours. What is its average speed?
<scratchpad>Speed = distance / time = 60 / 1.5 = 40.</scratchpad>
<answer>40 km/h</answer>

Q: A box has 24 apples. You remove one third. How many remain?
<scratchpad>One third of 24 = 8. Remaining = 24 - 8 = 16.</scratchpad>
<answer>16</answer>

Q: {new_problem}"""

# reasoning_effort=minimal 로 눌러 '모델 내부 추론'보다 '예시 포맷'의 효과가 드러나게 함
a = ask(prompt_A, reasoning_effort="minimal", max_completion_tokens=500)
b = ask(prompt_B, reasoning_effort="minimal", max_completion_tokens=500)
compare("A) 문제→정답 샷 (사고 노출 없음)", a, "B) scratchpad/answer 샷", b)

print("\n[관찰 포인트]")
print("- 정답: 12펜 = 4묶음 × $2 = $8 결제 → 거스름돈 = $10 - $8 = $2")
print("- B 출력에 <scratchpad>...</scratchpad><answer>...</answer> 구조가 이식됐는지 확인")
print("- 사고를 밖으로 꺼낸 B 쪽이 정답($2)에 더 안정적으로 도달하는지 비교")

## Tip 23 · 예시 과적합 판별 — 마커 단어가 새어 나오는지 보기

**요지.** 예시에 넣은 **특정 단어나 문구**가 최종 답변에서 그대로 반복된다면, 모델이 예시의 **형식이 아니라 내용까지 따라 배운**(내용 과적합) 신호다.

**무엇을 어떻게 검증하나.** 예시 답변 안에 실제로는 잘 쓰이지 않는 독특한 표시용 단어(마커) `blue-alpha`를 심어 둔다. 그런 다음 예시와 **아무 관련 없는** 새 질문(집중력을 유지하는 방법)을 던진다. `keyword_hits`로 이 마커가 답변에 등장하는지 개수를 센다 — **0이 바람직**하며, 이는 과적합이 없다는 뜻이다.

In [ ]:
# --- Tip 23 실험: 예시에 심은 마커가 무관한 답변에 새어나오는가? ---
MARKER = "blue-alpha"  # 현실에서 안 쓰는 독특한 마커 단어

prompt = f"""You are a helpful assistant. Here is one example of the desired answer style.

Example Q: What is a good name for a project mascot?
Example A: A memorable choice is "{MARKER}" — short, punchy, and easy to remember.

Now answer this NEW and unrelated question in the same helpful style:
Q: What are two practical tips for staying focused while studying?"""

out = ask(prompt, max_completion_tokens=500)
print(out)

# 마커/예시 소재(mascot, project)가 무관한 답변에 새어나왔는지 채점
score = keyword_hits(out, [MARKER, "mascot", "project"])
print("\n[과적합 채점]", score)
print(f"→ score 0 이 바람직: '{MARKER}' 등 예시 소재가 무관한 답변에 새어나오면 내용 과적합의 증거")

## Tip 26 · 구형 모델 내구성 테스트

**요지.** 값싼 **구형 모델에서도** 프롬프트가 의도대로 동작한다면, 그 프롬프트는 특정 모델의 높은 성능에 기대지 않는 **튼튼한 구조**를 갖췄다고 볼 수 있다.

**무엇을 어떻게 검증하나.** 역할·규칙·출력 형식을 분명하게 정해 둔 분류 프롬프트를 준비하고, 같은 프롬프트를 `MODEL`(gpt-5-nano)과 `OLD_MODEL`(gpt-4.1-nano)에 각각 `model=` 인자로 넘겨 실행한 뒤 `compare`로 결과를 비교한다.

> gpt-5-nano는 추론(reasoning) 모델이라 `reasoning_effort="minimal"`을 함께 주고, 구형 모델(gpt-4.1-nano)은 reasoning 관련 파라미터를 받지 않으므로 넘기지 않는다. 두 모델 모두 **답을 한 단어(대문자)로만 낸다는 규칙**을 지키면, 프롬프트 구조가 튼튼한 것이다.

In [ ]:
# --- Tip 26 실험: 잘 짜인 프롬프트를 신형 vs 구형에 각각 돌려 견고성 비교 ---
# 역할 + 규칙 + 출력형식을 못 박은 '방어적' 프롬프트
robust_prompt = """You are a strict text classifier.
Classify the sentiment of the review as exactly one of: POSITIVE, NEGATIVE, NEUTRAL.
Output ONLY that one word in uppercase. No explanation, no punctuation, no extra text.

Review: "The screen is gorgeous but the battery dies in three hours." """

# 신형: reasoning 모델 → reasoning_effort 지정 (본문 빈칸 방지용으로 토큰 넉넉히)
new = ask(robust_prompt, model=MODEL, reasoning_effort="minimal", max_completion_tokens=50)
# 구형: reasoning 파라미터 미지원 → 넘기지 않음
old = ask(robust_prompt, model=OLD_MODEL, max_completion_tokens=50)

compare(f"신형 {MODEL}", repr(new), f"구형 {OLD_MODEL}", repr(old))

print("\n[관찰 포인트]")
print("- 두 모델 모두 대문자 한 단어(예: NEGATIVE)만 내면 → 프롬프트 뼈대가 튼튼함")
print("- 구형이 설명을 덧붙이거나 형식을 깨면 → 규칙을 더 방어적으로 다듬어야 함")
# 규칙 준수(대문자 한 단어) 자동 채점
for label, txt in [("신형", new), ("구형", old)]:
    ok = (txt or "").strip() in {"POSITIVE", "NEGATIVE", "NEUTRAL"}
    print(f"  {label} 규칙 준수:", ok, "→", repr((txt or "").strip()))

## 요약 · 무엇을 보면 각 팁이 검증되는가

| 팁 | 실험 | 팁이 검증되는 관찰 포인트 |
| --- | --- | --- |
| **8** | 정보 추출 JSON, A=값이 채워진 예시 / B=빈 스키마+미러링 금지 | A가 없는 필드에 예시 값(A7788/50)을 가져오고, B는 `null`을 내면 → 값 편향 확인 |
| **9** | 산수 문제, A=문제→정답 / B=scratchpad/answer | B가 사고 과정 구조를 이어받아 정답($2)에 더 안정적으로 도달하면 → 추론 흐름 동기화 확인 |
| **23** | 마커 `blue-alpha`를 심은 예시 + 무관한 질문 | 답변에 마커가 등장하면(score>0) → 내용 과적합, 0이면 형식만 배운 것 |
| **26** | 같은 분류 프롬프트를 신형과 구형에서 실행 | 구형 모델까지 대문자 한 단어 규칙을 지키면 → 프롬프트 구조가 튼튼함 |

**핵심 교훈.** Few-shot 예시는 도움이 되기도 하고 해가 되기도 한다.
- 예시의 *값*은 그대로 따라 쓰인다(Tip 8) → 구조만 보여주고, 값은 따라 쓰지 말라고 분명히 적어라.
- 예시의 *형식*이 곧 출력 형식이 된다(Tip 9) → 사고 과정을 예시에 넣으면 추론도 그 형식을 따라온다.
- 예시의 *내용*이 답변에 새어 나오면 과적합이다(Tip 23) → 마커 단어로 누수 여부를 계속 점검하라.
- 좋은 프롬프트는 *모델에 덜 의존한다*(Tip 26) → 구형 모델에서도 통과하면 튼튼하다는 증거다.

> LLM 출력은 확률적이라 실행할 때마다 달라질 수 있다. 결과가 애매하면 여러 번 Run 해서 전체 경향을 보라.